# Scoring clusters with no labels

_Metrics & Evaluation — measuring models, data & statistics_

**Two kinds of cluster score: internal (is each cluster tight and well separated?) and external (does it match the known truth?).**

Clustering hands you groups but no grades. A score has to come from one of two places.

       INTERNAL metrics grade using only the points and their assigned cluster — no answer key. They all chase the same intuition: a good clustering has each cluster tight (members close together) and the clusters far apart (well separated). Tight + separated = good.

---

This notebook is a practice scaffold for the **Scoring clusters with no labels** lesson. We build the reference code one step at a time — run each cell, read the short note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_iris

data = load_iris(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Cluster the iris flowers with k-means

We load the iris data and run k-means asking for 3 clusters. `fit_predict` returns a cluster label for every flower. We also keep `y_true`, the real species — but those are **held out**; the clustering never sees them. Everything below scores the `labels` k-means produced.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans

iris = load_iris()
X, y_true = iris.data, iris.target          # y_true: the 3 real species (held out)

labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X)

### Step 2 — Internal scores (no labels needed)

Internal metrics use only `X` and the cluster `labels` — no answer key. The silhouette (higher is better, range -1..1) and Calinski–Harabasz (higher is better) reward tight, well-separated clusters; Davies–Bouldin is the odd one out where **lower is better**. These are what you reach for when you have no ground truth.

In [ ]:
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
)

# INTERNAL — is each cluster tight and well separated? (uses only X + labels)
print("silhouette        :", round(silhouette_score(X, labels), 3))        # higher better, [-1,1]
print("davies_bouldin    :", round(davies_bouldin_score(X, labels), 3))    # LOWER better, >= 0
print("calinski_harabasz :", round(calinski_harabasz_score(X, labels), 1)) # higher better

### Step 3 — External scores (compared to the true species)

External metrics compare the clustering to `y_true`. Adjusted Rand and AMI are **chance-corrected** (random labelings score ~0); NMI, the homogeneity/completeness/V triple, and Fowlkes–Mallows are other ways to measure agreement with the truth. Use these only when you actually have trusted labels to grade against.

In [ ]:
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    adjusted_mutual_info_score, homogeneity_completeness_v_measure,
    fowlkes_mallows_score,
)

# EXTERNAL — does the clustering match the true species? (compare labels to y_true)
print("adjusted_rand     :", round(adjusted_rand_score(y_true, labels), 3))            # chance-corrected
print("NMI               :", round(normalized_mutual_info_score(y_true, labels), 3))
print("AMI               :", round(adjusted_mutual_info_score(y_true, labels), 3))     # chance-corrected
h, c, v = homogeneity_completeness_v_measure(y_true, labels)
print("homog/compl/V     :", round(h, 3), round(c, 3), round(v, 3))
print("fowlkes_mallows   :", round(fowlkes_mallows_score(y_true, labels), 3))

# Real iris output:
# silhouette 0.553 | davies_bouldin 0.662 | calinski_harabasz 561.6
# ARI 0.73 | NMI 0.758 | AMI 0.755 | homog/compl/V 0.751 0.765 0.758 | FMI 0.821

## Visualize the data & results

_On the real iris flowers, as we vary the number of clusters k, where does the silhouette (no labels) peak versus the ARI (vs the true species)?_

### Step 1 — Sweep the number of clusters k

For each candidate `k` from 2 to 8 we refit k-means and record three things: the silhouette (internal, no labels), the ARI against the true species (external), and the inertia (within-cluster sum of squares). Collecting all three lets us compare what each signal *thinks* the right number of clusters is.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

iris = load_iris()
X, y_true = iris.data, iris.target          # 150 real flowers, 3 true species

ks = list(range(2, 9))
sil, ari, inertia = [], [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    labels = km.labels_
    sil.append(silhouette_score(X, labels))          # INTERNAL: no labels used
    ari.append(adjusted_rand_score(y_true, labels))  # EXTERNAL: vs the true species
    inertia.append(km.inertia_)                       # within-cluster sum of squares

for k, s, a, w in zip(ks, sil, ari, inertia):
    print(k, round(s, 3), round(a, 3), round(w, 1))

### Step 2 — Plot the scores against k

On the left, the silhouette (internal) peaks at **k=2** while the ARI (external) peaks at the true **k=3** — internal geometry and the actual truth can disagree. On the right, inertia drops at every k and never bottoms out, so you read its *elbow*, not its minimum.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left — internal vs external score as k changes
ax1.plot(ks, sil, "o-", color="#4ea1ff", label="silhouette (internal)")
ax1.plot(ks, ari, "o-", color="#7ee787", label="ARI vs truth (external)")
ax1.set_xlabel("number of clusters k")
ax1.set_ylabel("score")
ax1.legend()
ax1.set_title("silhouette peaks at k=2, ARI peaks at the true k=3")

# Right — inertia (WCSS) always falls; look for the elbow
ax2.bar([str(k) for k in ks], inertia, color="#ffb454")
ax2.set_xlabel("k")
ax2.set_ylabel("inertia (WCSS)")
ax2.set_title("inertia always drops — read the elbow, not the min")

plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You run $k$-means for $k=2,3,4,5$ and the within-cluster inertia is $152,\,79,\,57,\,46$. A teammate says "$k=5$ has the lowest inertia, so use 5 clusters." Why is that wrong, and what should you do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall how inertia behaves as $k$ grows. — _Adding a centroid can only keep points at an equal-or-closer center, so inertia can only drop — it is monotone in $k$._
- Notice the drops: $152\to79$ (big), $79\to57$ (smaller), $57\to46$ (small). — _The right $k$ is the elbow — where extra clusters stop buying much tightness — not the smallest value._
- Cross-check with a chance-aware internal score. — _The silhouette or gap statistic does not blindly reward more clusters, so it can actually pick $k$._

**Answer:** Lowest inertia never picks $k$ — inertia ALWAYS falls as $k$ rises and hits $0$ at $k=n$. The big drop is $152\to79$ (going from $2$ to $3$) and it flattens after, so the elbow is at $k=3$. Confirm with the silhouette / gap statistic, which (unlike inertia) penalize over-splitting. On the iris data below, the elbow and the truth both say $3$.

</details>

**Problem 2.** You cluster with no labels and your silhouette peaks at $k=2$, but you happen to know there are really $3$ true classes (and ARI vs the truth peaks at $k=3$). Which score do you trust, and why the disagreement?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify each metric's family. — _Silhouette is INTERNAL (tight & separated, no labels); ARI is EXTERNAL (matches the known truth)._
- Ask what each is optimizing. — _If two of the three true classes overlap, merging them into one blob looks tighter & more separated — so the silhouette prefers $k=2$._
- Decide based on whether you trust the labels. — _When ground truth exists and you trust it, "matches the truth" is the goal, so ARI wins._

**Answer:** Trust ARI ($k=3$) here, because you have ground truth and the question is correctness. The silhouette prefers $k=2$ because two of the three real classes overlap, so merging them yields a tighter, more-separated geometry — that is the classic "silhouette favors convex blobs" pitfall. Internal metrics answer "is it tight & separated?", which is not the same as "does it match the truth?" This exact split shows up on iris below.

</details>